# EDA — UNHCR Refugee X Dataset (URXD)

Analysis of the 250 manually labeled X posts provided by UNHCR.
This is the out-of-domain validation set — real recent data, very different from LMD.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

URXD_PATH = '../data/raw/urxd.csv'

if not os.path.exists(URXD_PATH):
    print('URXD data not found at', URXD_PATH)
    print('Place the file there and re-run')
else:
    df = pd.read_csv(URXD_PATH)
    print(f'Loaded {len(df)} rows')
    df.head()

In [ ]:
# standardize column names if needed
col_map = {}
for c in df.columns:
    if c.lower() in ['content', 'text', 'tweet']:
        col_map[c] = 'content'
    if c.lower() in ['label', 'verdict', 'class']:
        col_map[c] = 'label'
df = df.rename(columns=col_map)

# normalize labels
df['label'] = df['label'].apply(lambda x: 1 if x in [1, '1', 'TRUE', 'True', 'true', True] else 0)

print(f'Total: {len(df)} posts')
print(f'Class distribution: {df["label"].value_counts().to_dict()}')

## Label Distribution

Only 250 samples — this is a deliberately small but high-quality validation set.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts()
label_names = {0: 'Misinformation', 1: 'Factual'}

bars = axes[0].bar(
    [label_names[i] for i in counts.index],
    counts.values,
    color=['#e74c3c', '#2ecc71']
)
axes[0].set_title('URXD Class Distribution')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(int(bar.get_height())), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=[label_names[i] for i in counts.index],
            autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'])
axes[1].set_title('URXD Class Split')

plt.suptitle('UNHCR Refugee X Dataset — 250 manually labeled posts', fontsize=10)
plt.tight_layout()
plt.show()

## Text Length

In [ ]:
df['word_count'] = df['content'].apply(lambda x: len(str(x).split()))
df['char_count'] = df['content'].apply(len)

print('Text length statistics:')
print(df[['word_count', 'char_count']].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, name, color in [(0, 'Misinformation', '#e74c3c'), (1, 'Factual', '#2ecc71')]:
    subset = df[df['label'] == label]
    axes[0].hist(subset['word_count'], bins=20, alpha=0.6, label=name, color=color)
    axes[1].hist(subset['char_count'], bins=20, alpha=0.6, label=name, color=color)

axes[0].set_title('Word Count by Class')
axes[0].set_xlabel('Words')
axes[0].legend()

axes[1].set_title('Character Count by Class')
axes[1].set_xlabel('Characters')
axes[1].legend()

plt.tight_layout()
plt.show()

## Sentiment Comparison

In [ ]:
try:
    import nltk
    from nltk.sentiment import SentimentIntensityAnalyzer
    nltk.download('vader_lexicon', quiet=True)
    sia = SentimentIntensityAnalyzer()
    df['sentiment'] = df['content'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

    fig, ax = plt.subplots(figsize=(10, 4))
    df_plot = df[['label', 'sentiment']].copy()
    df_plot['class'] = df_plot['label'].map({0: 'Misinformation', 1: 'Factual'})
    sns.violinplot(data=df_plot, x='class', y='sentiment', ax=ax,
                   palette={'Misinformation': '#e74c3c', 'Factual': '#2ecc71'})
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title('Sentiment Polarity — URXD')
    plt.tight_layout()
    plt.show()

    print('Mean sentiment:')
    print(df.groupby('label')['sentiment'].mean().round(3))
except ImportError:
    print('Install nltk for sentiment analysis: pip install nltk')

## Common Topics and Bigrams

In [ ]:
def get_bigrams(texts, n=20):
    stopwords = {'the','a','an','is','are','was','to','of','in','for','on',
                 'and','or','but','not','this','that','it','i','he','she','they',
                 'we','url','hashtag','mention','rt','with','at','by','from'}
    bigrams = []
    for t in texts:
        words = [w for w in str(t).lower().split() if w not in stopwords and len(w) > 2]
        bigrams += [(words[i], words[i+1]) for i in range(len(words)-1)]
    return Counter(bigrams).most_common(n)

misinfo_bigrams = get_bigrams(df[df['label']==0]['content'].tolist())
factual_bigrams = get_bigrams(df[df['label']==1]['content'].tolist())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for bigrams, ax, title, color in [
    (misinfo_bigrams, axes[0], 'Top Bigrams — Misinformation', '#e74c3c'),
    (factual_bigrams, axes[1], 'Top Bigrams — Factual', '#2ecc71')
]:
    labels_b = [f'{a} {b}' for (a,b),_ in bigrams[:15]]
    counts_b = [c for _,c in bigrams[:15]]
    ax.barh(labels_b[::-1], counts_b[::-1], color=color, alpha=0.8)
    ax.set_title(title)

plt.tight_layout()
plt.show()

## Comparison with LMD

In [ ]:
lmd_path = '../data/processed/twitter_filtered/train.json'
if os.path.exists(lmd_path):
    lmd = pd.read_json(lmd_path)
    lmd['word_count'] = lmd['content'].apply(lambda x: len(str(x).split()))

    comparison = pd.DataFrame({
        'Metric': ['N samples', 'Avg word count', 'Misinfo %', 'Avg char count'],
        'URXD': [
            len(df),
            df['word_count'].mean().round(1),
            f"{100*df['label'].mean():.1f}%",
            df['char_count'].mean().round(0),
        ],
        'LMD (twitter_filtered)': [
            len(lmd),
            lmd['word_count'].mean().round(1),
            f"{100*(1-lmd['label'].mean()):.1f}%" if 'label' in lmd.columns else 'N/A',
            lmd['char_count'].mean().round(0) if 'char_count' in lmd.columns else 'N/A',
        ]
    })
    print(comparison.to_string(index=False))
else:
    print('LMD not found for comparison')

## Key Observations

- **Small but targeted**: 250 samples, all specifically about refugees/migrants on X — this is the hardest test for the model
- **More balanced** than some LMD configurations
- **Different distribution**: URXD posts tend to be shorter (tweet format) and more recent (2024)
- **Topic specificity**: Refugee/migrant content involves specific narratives (border crossings, crime claims, economic impact) not always well-represented in general misinfo datasets
- **Domain shift**: Models trained on LMD face a real domain shift challenge when evaluated on URXD — which is why the distilled model with domain adaptation performs better than raw fine-tuning

## Annotation Notes

The 250 posts were labeled manually by the thesis authors following UNHCR's definition of misinformation. Each post was reviewed independently and discussed with UNHCR contacts (Kwabena Adwabour and John Bastawrous).

Labeling criteria:
- **Misinformation (0)**: Contains false or misleading claims about refugees/migrants, regardless of intent
- **Factual (1)**: Factually accurate, even if critical or negative in tone

Difficult edge cases included: satire, decontextualized facts, opinion posts, and ambiguous claims.